# Skew Detection & Affine Rectification Engine

**The mechanistic justification for this step is:** Provide automated fabric orientation alignment using frequency-domain polar Radon integration and inverse affine rotation, preventing projection profile collapse and restoring yarn periodicity for accurate warp and weft counting.

In [1]:
# The mechanistic justification for this step is: configure the global Kaggle pathing invariant with local fallback routing and initialize PyTorch hardware execution.
import os
import json
import numpy as np
import torch
from PIL import Image

INPUT_DIR = "/kaggle/input/fabric-data" if os.path.exists("/kaggle/input/fabric-data") else "data"
WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "results"
os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(os.path.join(INPUT_DIR, "synthetic"), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Allocated Memory (MB): {torch.cuda.memory_allocated() / (1024 * 1024):.2f}")
print(f"Input Directory: {INPUT_DIR}")
print(f"Working Directory: {WORKING_DIR}")

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Allocated Memory (MB): 0.00
Input Directory: data
Working Directory: results


In [2]:
# The mechanistic justification for this step is: generate synthetic macro plain-weave fabric with controlled, non-orthogonal angular tilt (theta = +15.0 deg) and known yarn density (Warp: 32 ends/cm, Weft: 28 picks/cm) to serve as a verifiable benchmark for skew estimation and rectification.
pixels_per_cm = 400
img_size = 400
true_warp = 32
true_weft = 28
true_skew_deg = 15.0

x = np.linspace(0, 1, img_size, endpoint=False)
y = np.linspace(0, 1, img_size, endpoint=False)
X, Y = np.meshgrid(x, y)

warp_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_warp * X))
weft_signal = 0.5 * (1.0 + np.cos(2.0 * np.pi * true_weft * Y))
interlace = np.sin(2.0 * np.pi * true_warp * X) * np.sin(2.0 * np.pi * true_weft * Y)
base_texture = 0.45 * warp_signal + 0.45 * weft_signal + 0.1 * interlace

np.random.seed(42)
noise = np.random.normal(0, 0.05, (img_size, img_size))
aligned_fabric = np.clip(base_texture + noise, 0.0, 1.0)
aligned_pil = Image.fromarray((aligned_fabric * 255).astype(np.uint8))

skewed_pil = aligned_pil.rotate(true_skew_deg, resample=Image.Resampling.BICUBIC)
skewed_fabric = np.array(skewed_pil, dtype=np.float32) / 255.0
skewed_path = os.path.join(INPUT_DIR, "synthetic", "sample_skewed_15deg.png")
skewed_pil.save(skewed_path)

print(f"Fabric Shape: {skewed_fabric.shape}")
print(f"True Warp (ends/cm): {true_warp}")
print(f"True Weft (picks/cm): {true_weft}")
print(f"True Skew (deg): {true_skew_deg}")

Fabric Shape: (400, 400)
True Warp (ends/cm): 32
True Weft (picks/cm): 28
True Skew (deg): 15.0


In [3]:
# The mechanistic justification for this step is: transform the skewed spatial domain fabric into the frequency domain via 2D Fast Fourier Transform and perform angular polar integration across [-45 deg, +45 deg] to isolate the dominant yarn normal axis.
tensor_fabric = torch.tensor(skewed_fabric, device=device, dtype=torch.float32)
mean_val = torch.mean(tensor_fabric)
f_input = tensor_fabric - mean_val

fft2 = torch.fft.fft2(f_input)
fft_shift = torch.fft.fftshift(fft2)
magnitude = torch.abs(fft_shift)

rows, cols = magnitude.shape
crow, ccol = rows // 2, cols // 2

angles = np.linspace(-45, 45, 181)
angular_energies = []

r_min = 15
r_max = int(min(rows, cols) * 0.45)
r_vals = torch.arange(r_min, r_max, device=device, dtype=torch.float32)

for theta in angles:
    rad = np.deg2rad(theta)
    # Coordinate alignment: row index increases downwards; CCW visual rotation shifts row upward (-sin)
    u_idx = torch.clamp(torch.round(crow - r_vals * np.sin(rad)).long(), 0, rows - 1)
    v_idx = torch.clamp(torch.round(ccol + r_vals * np.cos(rad)).long(), 0, cols - 1)
    energy = torch.sum(magnitude[u_idx, v_idx]).item()
    angular_energies.append(energy)

angular_energies = np.array(angular_energies)
detected_skew_deg = float(angles[np.argmax(angular_energies)])
skew_error_deg = abs(detected_skew_deg - true_skew_deg)

print(f"Detected Skew Angle (deg): {detected_skew_deg}")
print(f"Ground Truth Skew Angle (deg): {true_skew_deg}")
print(f"Skew Estimation Error (deg): {skew_error_deg}")

Detected Skew Angle (deg): 15.0
Ground Truth Skew Angle (deg): 15.0
Skew Estimation Error (deg): 0.0


In [4]:
# The mechanistic justification for this step is: apply inverse affine rotation by -theta_detected using bicubic interpolation and crop out boundary zero-padding triangles to ensure pure textile texture for subsequent analysis.
rectified_pil = skewed_pil.rotate(-detected_skew_deg, resample=Image.Resampling.BICUBIC)
rectified_fabric = np.array(rectified_pil, dtype=np.float32) / 255.0
rectified_path = os.path.join(WORKING_DIR, "rectified_sample.png")
rectified_pil.save(rectified_path)

margin = int(img_size * 0.15)
skewed_crop = skewed_fabric[margin : img_size - margin, margin : img_size - margin]
rectified_crop = rectified_fabric[margin : img_size - margin, margin : img_size - margin]

print(f"Original Dimensions: {skewed_fabric.shape}")
print(f"Rectified Crop Dimensions: {rectified_crop.shape}")
print(f"Cropped Boundary Margin (px): {margin}")

Original Dimensions: (400, 400)
Rectified Crop Dimensions: (280, 280)
Cropped Boundary Margin (px): 60


In [5]:
# The mechanistic justification for this step is: compute marginal 1D spatial projection profiles across the skewed and rectified fabrics to evaluate contrast recovery ratio and peak prominence restoration.
skewed_warp_prof = np.mean(skewed_crop, axis=0)
rectified_warp_prof = np.mean(rectified_crop, axis=0)

skewed_contrast = float(np.std(skewed_warp_prof))
rectified_contrast = float(np.std(rectified_warp_prof))
contrast_gain_ratio = float(rectified_contrast / (skewed_contrast + 1e-8))

print(f"Skewed Projection StdDev: {skewed_contrast:.6f}")
print(f"Rectified Projection StdDev: {rectified_contrast:.6f}")
print(f"Contrast Gain Ratio: {contrast_gain_ratio:.2f}x")

Skewed Projection StdDev: 0.005678
Rectified Projection StdDev: 0.163332
Contrast Gain Ratio: 28.77x


In [6]:
# The mechanistic justification for this step is: execute 2D FFT spectral peak extraction with sub-bin parabolic interpolation on the rectified ROI to determine yarn density with fractional precision.
rect_tensor = torch.tensor(rectified_crop, device=device, dtype=torch.float32)
rect_f = rect_tensor - torch.mean(rect_tensor)
rect_mag = torch.abs(torch.fft.fftshift(torch.fft.fft2(rect_f))).cpu().numpy()

rcrow, rccol = rect_mag.shape[0] // 2, rect_mag.shape[1] // 2
crop_scale = rect_mag.shape[1] / img_size
max_radial_bins = int(rect_mag.shape[1] * 0.45)

horiz_spec = np.mean(rect_mag[rcrow - 2 : rcrow + 3, rccol + 3 : rccol + max_radial_bins], axis=0)
vert_spec = np.mean(rect_mag[rcrow + 3 : rcrow + max_radial_bins, rccol - 2 : rccol + 3], axis=1)

def refine_peak(spec, peak_idx):
    if 0 < peak_idx < len(spec) - 1:
        alpha = spec[peak_idx - 1]
        beta = spec[peak_idx]
        gamma = spec[peak_idx + 1]
        denom = alpha - 2 * beta + gamma
        if abs(denom) > 1e-8:
            delta = 0.5 * (alpha - gamma) / denom
            return float(peak_idx) + delta
    return float(peak_idx)

horiz_raw_idx = np.argmax(horiz_spec)
refined_horiz_idx = refine_peak(horiz_spec, horiz_raw_idx)
pred_warp_rect_fft = float(round(((refined_horiz_idx + 3) / crop_scale), 1))

vert_raw_idx = np.argmax(vert_spec)
refined_vert_idx = refine_peak(vert_spec, vert_raw_idx)
pred_weft_rect_fft = float(round(((refined_vert_idx + 3) / crop_scale), 1))

delta_warp = abs(pred_warp_rect_fft - true_warp)
delta_weft = abs(pred_weft_rect_fft - true_weft)

print(f"Predicted Warp Ends/cm: {pred_warp_rect_fft} (Ground Truth: {true_warp}) | Delta: {delta_warp:.2f}")
print(f"Predicted Weft Picks/cm: {pred_weft_rect_fft} (Ground Truth: {true_weft}) | Delta: {delta_weft:.2f}")

Predicted Warp Ends/cm: 31.700000762939453 (Ground Truth: 32) | Delta: 0.30
Predicted Weft Picks/cm: 28.299999237060547 (Ground Truth: 28) | Delta: 0.30


In [7]:
# The mechanistic justification for this step is: generate a comprehensive 6-panel diagnostic visualization comparing the skewed fabric, 2D FFT spectrum, Radon energy profile, rectified fabric, projection restoration, and harmonic spectrum, and persist the figure to disk.
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

axes[0, 0].imshow(skewed_fabric, cmap='gray')
axes[0, 0].set_title(f'Skewed Fabric (+{true_skew_deg} deg)', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(np.log1p(magnitude.cpu().numpy()), cmap='inferno')
axes[0, 1].set_title('2D FFT Spectrum (Skewed)', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].plot(angles, angular_energies, color='darkblue', lw=2)
axes[0, 2].axvline(detected_skew_deg, color='red', linestyle='--', label=f'Peak @ {detected_skew_deg} deg')
axes[0, 2].set_title('Polar Radon Scan Energy vs Angle', fontsize=11, fontweight='bold')
axes[0, 2].set_xlabel('Angle (degrees)')
axes[0, 2].set_ylabel('Radial Spectral Energy')
axes[0, 2].legend(loc='upper right')
axes[0, 2].grid(True, alpha=0.3)

axes[1, 0].imshow(rectified_fabric, cmap='gray')
axes[1, 0].set_title(f'Rectified Fabric (-{detected_skew_deg} deg)', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].plot(skewed_warp_prof, label=f'Skewed (std={skewed_contrast:.4f})', color='gray', linestyle=':')
axes[1, 1].plot(rectified_warp_prof, label=f'Rectified (std={rectified_contrast:.4f})', color='green', lw=1.5)
axes[1, 1].set_title(f'Projection Profile: Contrast Gain {contrast_gain_ratio:.1f}x', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Pixel Column Index')
axes[1, 1].set_ylabel('Mean Intensity')
axes[1, 1].legend(loc='upper right')
axes[1, 1].grid(True, alpha=0.3)

freq_warp = (np.arange(len(horiz_spec)) + 3) / crop_scale
axes[1, 2].plot(freq_warp, horiz_spec, color='purple', lw=1.5, label='Warp Harmonic')
axes[1, 2].axvline(pred_warp_rect_fft, color='red', linestyle='--', label=f'Warp: {pred_warp_rect_fft} ends/cm')
axes[1, 2].set_title('Rectified Harmonic Spectrum (Warp)', fontsize=11, fontweight='bold')
axes[1, 2].set_xlabel('Frequency (cycles / cm)')
axes[1, 2].set_ylabel('Spectral Amplitude')
axes[1, 2].legend(loc='upper right')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(WORKING_DIR, 'skew_rectification_comparison.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Artifact Saved: {fig_path}")

Artifact Saved: results/skew_rectification_comparison.png


In [8]:
# The mechanistic justification for this step is: persist structured empirical metrics to results/output_skew.json and human-readable summary to results/output_skew.txt for auditability and provenance ledger tracking.
results_payload = {
    "true_skew_degrees": true_skew_deg,
    "detected_skew_degrees": detected_skew_deg,
    "skew_estimation_error_degrees": skew_error_deg,
    "skewed_projection_variance": skewed_contrast,
    "rectified_projection_variance": rectified_contrast,
    "contrast_recovery_ratio": contrast_gain_ratio,
    "rectified_warp_fft": pred_warp_rect_fft,
    "rectified_weft_fft": pred_weft_rect_fft,
    "ground_truth_warp": true_warp,
    "ground_truth_weft": true_weft,
    "delta_warp": delta_warp,
    "delta_weft": delta_weft
}

with open(os.path.join(WORKING_DIR, "output_skew.json"), "w") as f:
    json.dump(results_payload, f, indent=2)

with open(os.path.join(WORKING_DIR, "output_skew.txt"), "w") as f:
    f.write("=== SKEW RECTIFICATION & RESTORATION REPORT ===\n")
    f.write(f"True Skew: {true_skew_deg} deg | Detected Skew: {detected_skew_deg} deg | Error: {skew_error_deg} deg\n")
    f.write(f"Projection Contrast Gain: {contrast_gain_ratio:.2f}x\n")
    f.write(f"Rectified Pred Warp: {pred_warp_rect_fft} (True: {true_warp}) | Pred Weft: {pred_weft_rect_fft} (True: {true_weft})\n")

print(json.dumps(results_payload, indent=2))

{
  "true_skew_degrees": 15.0,
  "detected_skew_degrees": 15.0,
  "skew_estimation_error_degrees": 0.0,
  "skewed_projection_variance": 0.005678019020706415,
  "rectified_projection_variance": 0.16333185136318207,
  "contrast_recovery_ratio": 28.765589391591664,
  "rectified_warp_fft": 31.700000762939453,
  "rectified_weft_fft": 28.299999237060547,
  "ground_truth_warp": 32,
  "ground_truth_weft": 28,
  "delta_warp": 0.2999992370605469,
  "delta_weft": 0.2999992370605469
}
